In [ ]:
# ============================================================
# MEDIVOICE - Gemma 4 Good Hackathon | github.com/hamnamgl/Medivoice
# Offline AI Health Copilot for Frontline Community Health Workers
# ============================================================
import json, os, sys, threading, time, requests, subprocess
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import display, HTML

print("NOTE: This Kaggle notebook is a reproducible demo environment.")
print("NOTE: True offline deployment runs the same MediVoice stack locally with Ollama on-device.")

# -- 1. Clone / update repo ----------------------------------
REPO = "/kaggle/working/Medivoice"
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/hamnamgl/Medivoice.git {REPO}")
    print("[ok] Repo cloned")
else:
    os.system(f"git -C {REPO} pull")
    print("[ok] Repo updated")
sys.path.insert(0, REPO)
os.chdir(REPO)

# -- 2. Install Python dependencies --------------------------
subprocess.run(
    "pip install -q ollama openai-whisper edge-tts requests pyngrok > /tmp/medivoice_pip.log 2>&1",
    shell=True,
    check=False,
)
print("[ok] Dependencies installed")

# -- 3. Install zstd (required by Ollama installer) ---------
subprocess.run(
    "apt-get install -y -q zstd > /tmp/medivoice_apt.log 2>&1",
    shell=True,
    check=False,
)
print("[ok] zstd installed")

# -- 4. Install Ollama (only if binary missing) -------------
OLLAMA_BIN = "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA_BIN):
    print("Installing Ollama...")
    ret = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh > /tmp/medivoice_ollama_install.log 2>&1",
        shell=True,
        check=False,
    )
    if ret.returncode == 0 and os.path.exists(OLLAMA_BIN):
        print("[ok] Ollama installed")
    else:
        print("[error] Ollama install failed - check /tmp/medivoice_ollama_install.log")
else:
    print("[ok] Ollama already installed - skipping")

# -- 5. Start Ollama server ---------------------------------
def start_ollama():
    os.system("ollama serve > /tmp/ollama.log 2>&1")

threading.Thread(target=start_ollama, daemon=True).start()

print("Waiting for Ollama server to start...")
for _ in range(30):
    try:
        r = requests.get("http://localhost:11434/", timeout=2)
        if r.status_code == 200:
            print("[ok] Ollama server is running")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("[error] Ollama server did not start in time")

# -- 6. Pull model only if not already cached ---------------
MODEL = "gemma3:4b"

def model_is_cached(model_name):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = r.json().get("models", [])
        return any(m.get("name", "") == model_name for m in models)
    except Exception:
        return False

if model_is_cached(MODEL):
    print(f"[ok] {MODEL} already cached - skipping download")
else:
    print(f"Downloading {MODEL} - only once, cached after this...")
    pull_result = subprocess.run(
        f"ollama pull {MODEL} > /tmp/medivoice_ollama_pull.log 2>&1",
        shell=True,
        check=False,
    )
    if pull_result.returncode == 0:
        print("[ok] Model downloaded and cached")
    else:
        print("[error] Model download failed. Showing last Ollama pull log lines:")
        os.system("tail -20 /tmp/medivoice_ollama_pull.log")

# -- 7. Smoke test ------------------------------------------
print("Running smoke test...")
try:
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": "Say exactly: Medi is ready"}],
            "stream": False,
            "options": {"num_predict": 20},
        },
        timeout=60,
    )
    reply = r.json().get("message", {}).get("content", "")
    print(f"[ok] Model working - Medi: {reply}")
except Exception as e:
    print(f"[error] Ollama error: {e}")
    print("Ollama log tail:")
    os.system("tail -20 /tmp/ollama.log")

# -- 8. Start a CORS-safe local proxy for browser/PWA access -
PROXY_PORT = 8000

class OllamaProxyHandler(BaseHTTPRequestHandler):
    def _set_headers(self, status_code=200, content_type="application/json"):
        self.send_response(status_code)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.send_header("Content-Type", content_type)
        self.end_headers()

    def do_OPTIONS(self):
        self._set_headers(204, "text/plain")

    def do_GET(self):
        if self.path in ("/", "/health"):
            self._set_headers(200)
            self.wfile.write(json.dumps({"status": "ok", "service": "medivoice-proxy"}).encode("utf-8"))
            return
        if self.path == "/api/tags":
            try:
                upstream = requests.get("http://localhost:11434/api/tags", timeout=30)
                self._set_headers(upstream.status_code)
                self.wfile.write(upstream.content)
            except Exception as exc:
                self._set_headers(502)
                self.wfile.write(json.dumps({"error": str(exc)}).encode("utf-8"))
            return
        self._set_headers(404)
        self.wfile.write(json.dumps({"error": "Not found"}).encode("utf-8"))

    def do_POST(self):
        if self.path != "/api/chat":
            self._set_headers(404)
            self.wfile.write(json.dumps({"error": "Not found"}).encode("utf-8"))
            return
        try:
            content_length = int(self.headers.get("Content-Length", "0"))
            body = self.rfile.read(content_length)
            upstream = requests.post(
                "http://localhost:11434/api/chat",
                data=body,
                headers={"Content-Type": self.headers.get("Content-Type", "application/json")},
                timeout=180,
            )
            self._set_headers(upstream.status_code)
            self.wfile.write(upstream.content)
        except Exception as exc:
            self._set_headers(502)
            self.wfile.write(json.dumps({"error": str(exc)}).encode("utf-8"))

    def log_message(self, format, *args):
        return

proxy_server = ThreadingHTTPServer(("0.0.0.0", PROXY_PORT), OllamaProxyHandler)
threading.Thread(target=proxy_server.serve_forever, daemon=True).start()
print(f"[ok] Browser-safe proxy running on port {PROXY_PORT}")

# -- 9. Public URL for PWA (Option 1: ngrok) ----------------
OLLAMA_URL = None
NGROK_ERROR = None
ngrok_token = ""

try:
    from pyngrok import ngrok

    ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    if not ngrok_token:
        try:
            from kaggle_secrets import UserSecretsClient
            ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN").strip()
            print("[ok] NGROK_AUTHTOKEN loaded from Kaggle Secrets")
        except Exception:
            ngrok_token = ""

    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print("[ok] NGROK_AUTHTOKEN found and configured")
    else:
        print("[info] NGROK_AUTHTOKEN not found")
        print("[info] In Kaggle, add it under Add-ons -> Secrets with exact name NGROK_AUTHTOKEN, then restart and rerun the cell.")

    public_url = ngrok.connect(PROXY_PORT, bind_tls=True)
    OLLAMA_URL = public_url.public_url.rstrip("/") + "/api/chat"
    print(f"[ok] Browser-safe Ollama Public URL: {OLLAMA_URL}")
    print("Paste this exact URL into the PWA Custom Ollama URL field.")
except Exception as e:
    NGROK_ERROR = str(e)
    print("[warn] ngrok public tunnel could not be created.")
    print("If you want PWA access from phone/browser, set NGROK_AUTHTOKEN in Kaggle secrets/environment and rerun.")
    print(f"ngrok error: {NGROK_ERROR}")

# -- 10. Live consultation demo -----------------------------
from app.core.function_caller import run_agent
from app.core.triage_logic import assess_severity
from app.utils.local_db import init_db, get_stats, log_visit
init_db()

tests = [
    ("English", "Child has high fever for 3 days and not eating"),
    ("Roman Urdu", "Bachche ko 3 din se tez bukhar hai"),
    ("Hausa", "Yaro yana da zazzabi tsawon kwanaki 3"),
    ("Tool Test", "15kg child - paracetamol dosage?"),
    ("Tool Test", "Nearest hospital in Punjab?"),
    ("Emergency", "Patient is unconscious and not breathing"),
]

print("\n" + "=" * 58)
print("MEDIVOICE - LIVE CONSULTATION DEMO")
print("Offline AI | Gemma 4 | Ollama | 22 Languages")
print("=" * 58)

history = []
for label, msg in tests:
    print(f"\n[{label}]")
    print(f"CHW   : {msg}")
    result = run_agent(msg, history)
    history = result["history"]
    print(f"Medi  : {result['response']}")
    if result["tool_used"]:
        print(f"Tool  : {result['tool_used']}")
        print(f"Result: {result['tool_result']}")
    severity = assess_severity(msg)
    if result.get("tool_used") == "lookup_referral":
        severity = {"level": "REFER", "action": "CLINIC REFER KAREIN"}
    elif result.get("tool_used") == "get_drug_dosage":
        severity = {"level": "HOME CARE", "action": "GHAR PE DEKHBHAL"}
    log_visit(
        symptoms=msg,
        severity=severity["level"],
        action=severity["action"],
        language=label.lower(),
        tool_used=result.get("tool_used"),
        response=result["response"],
    )
    print("-" * 40)

# -- 11. Stats ----------------------------------------------
s = get_stats()
print(f"\nSession Stats - Total:{s['total_visits']} | Emergency:{s['emergencies']} | Refer:{s['referrals']} | Home:{s['home_care']}")

# -- 12. PWA link -------------------------------------------
if OLLAMA_URL:
    pwa_mode = f"""
    <div style=\"background:#0a1a0a;border-radius:8px;padding:14px;
                font-size:0.82rem;color:#86efac;border-left:3px solid #4ade80;\">
      <b>Use with Kaggle + ngrok proxy:</b><br><br>
      1. Open <b>https://hamnamgl.github.io/Medivoice</b><br>
      2. Paste this exact URL into <b>Custom Ollama URL</b><br>
      3. <code>{OLLAMA_URL}</code><br>
      4. Tap <b>Save URL</b> and start chatting<br><br>
      This browser-safe proxy avoids the direct cross-origin Ollama issue.
    </div>
    """
else:
    pwa_mode = f"""
    <div style=\"background:#2a2210;border-radius:8px;padding:14px;
                font-size:0.82rem;color:#fde68a;border-left:3px solid #fbbf24;\">
      <b>ngrok setup required for phone/browser demo:</b><br><br>
      1. Create and verify an ngrok account<br>
      2. Add <code>NGROK_AUTHTOKEN</code> to Kaggle secrets/environment<br>
      3. Restart the notebook session and rerun this cell<br><br>
      <b>Current ngrok error:</b><br>
      <code>{NGROK_ERROR}</code>
    </div>
    """

display(HTML(f"""
<div style="font-family:sans-serif;background:#1a1a2e;color:#e0e0e0;
     padding:24px;border-radius:12px;border:1px solid #2a4a6a;margin:16px 0;">
  <h2 style="color:#7eb8f7;margin-bottom:8px;">MediVoice PWA - Install on Android</h2>
  <p style="color:#a0c0a0;font-size:0.85rem;margin-bottom:16px;">
    Kaggle notebook = reproducible demo. True offline mode runs locally with Ollama on-device.
  </p>
  <a href="https://hamnamgl.github.io/Medivoice" target="_blank"
     style="display:inline-block;background:#4ade80;color:#0f0f1a;
            padding:12px 32px;border-radius:24px;font-weight:bold;
            text-decoration:none;font-size:1rem;margin-bottom:20px;">
    Open MediVoice PWA
  </a>
  {pwa_mode}
  <div style="margin-top:14px;font-size:0.78rem;color:#7eb8f7;">
    Repo: <a href="https://github.com/hamnamgl/Medivoice"
             style="color:#4ade80;">github.com/hamnamgl/Medivoice</a>
  </div>
</div>
"""))

print("\nMediVoice demo complete.")
print("Repo : https://github.com/hamnamgl/Medivoice")
print("PWA  : https://hamnamgl.github.io/Medivoice")
if OLLAMA_URL:
    print(f"Tunnel: {OLLAMA_URL}")
else:
    print("Tunnel: not available - set NGROK_AUTHTOKEN and rerun for public PWA access")
